# mpH2MM Analysis — PTU files (Alexa555 / Alexa647)

**Fixes applied vs original notebook:**
| Parameter | Original (wrong) | Fixed (from reference scripts) |
|---|---|---|
| Background fit | `exp_fit`, 30s | `exp_hist_fit`, 60s, F_bg=1.7 |
| Burst search m | 20 | 15 |
| Ph_sel | all photons | `Ph_sel(Dex='DAem')` |
| min_burst_size | 60 | 150 |
| min_naa | 40 | 30 |
| S filter | missing | S ∈ [0.25, 0.85] |
| BVA chunk size | 5 | 7 |
| Donor-only filter | wrong burst_type bitmask | correct `burst_type` logic |


## 1 · Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import fretbursts as frb
import phconvert as phc
import burstH2MM as bhm
import H2MM_C as h2

print('NumPy      :', np.__version__)
print('FRETbursts :', frb.__version__)
print('phconvert  :', phc.__version__)
print('burstH2MM  :', bhm.__version__)
print('H2MM_C     :', h2.__version__)


## 2 · PTU → Photon-HDF5 converter
Uses `phc.loader.nsalex_pq` (correct high-level ns-ALEX loader).  
`donor=1` (Alexa555), `acceptor=0` (Alexa647), ALEX periods (0,1000)/(1000,2000).


In [ ]:
def ptu_to_hdf5(
    ptu_file: str,
    output_path: str,
    description: str = 'smFRET — Alexa555/Alexa647',
    author: str = 'Fatma',
    author_affiliation: str = 'KUL',
    sample_name: str = '',
    buffer_name: str = 'Tris',
    dye_names: str = 'Alexa555, Alexa647',
):
    print(f'Reading {ptu_file} ...')
    d, meta = phc.loader.nsalex_pq(
        str(ptu_file),
        donor=1,           # Alexa555 emission → detector 1
        acceptor=0,        # Alexa647 emission → detector 0
        alex_period_donor=(0, 1000),
        alex_period_acceptor=(1000, 2000),
        excitation_wavelengths=(520e-9, 640e-9),
        detection_wavelengths=(570e-9, 690e-9),
    )

    nanotimes  = d['photon_data']['nanotimes']
    detectors  = d['photon_data']['detectors']
    timestamps = d['photon_data']['timestamps']

    # Remove overflow photons (nanotimes == 0 are sync/IRF artifacts)
    not_overflow = nanotimes != 0
    print(f'Removing {(~not_overflow).sum()} overflow photons '
          f'({(~not_overflow).mean()*100:.2f}%)')
    d['photon_data']['nanotimes']  = nanotimes[not_overflow]
    d['photon_data']['detectors']  = detectors[not_overflow]
    d['photon_data']['timestamps'] = timestamps[not_overflow]

    unique_det, counts = np.unique(d['photon_data']['detectors'], return_counts=True)
    print('Detectors after filter:', dict(zip(unique_det.tolist(), counts.tolist())))

    # TCSPC decay plot — verify ALEX period windows are correct
    fig, ax = plt.subplots(figsize=(7, 3))
    for det_id in unique_det:
        mask = d['photon_data']['detectors'] == det_id
        ax.hist(d['photon_data']['nanotimes'][mask], bins=256,
                histtype='step', label=f'det {det_id}')
    ax.axvspan(0,    1000, alpha=0.15, color='green', label='donor window (0-1000)')
    ax.axvspan(1000, 2000, alpha=0.15, color='red',   label='acceptor window (1000-2000)')
    ax.set_yscale('log')
    ax.set_xlabel('nanotime (ticks)')
    ax.set_ylabel('counts')
    ax.legend(fontsize=8)
    ax.set_title('TCSPC decays — verify ALEX windows match the peaks')
    plt.tight_layout(); plt.show()

    d['description'] = description
    d['sample'] = dict(
        sample_name=sample_name, dye_names=dye_names,
        buffer_name=buffer_name, num_dyes=len(dye_names.split(',')),
    )
    d['identity'] = dict(author=author, author_affiliation=author_affiliation)
    _ = meta.pop('dispcurve', None)
    _ = meta.pop('imghdr', None)
    d['user'] = {'picoquant': meta}

    phc.hdf5.save_photon_hdf5(d, overwrite=True)
    actual = str(Path(ptu_file).with_suffix('.hdf5'))
    print(f'Saved → {actual}')


## 3 · Filter junk detectors (keep det 0 & 1 only)

In [ ]:
import shutil, h5py

def filter_detectors(input_h5: str, output_h5: str, keep: tuple = (0, 1)):
    """Keep only photons from detectors in `keep`, remove junk channels."""
    shutil.copy(input_h5, output_h5)
    with h5py.File(output_h5, 'r+') as f:
        det  = f['photon_data/detectors'][:]
        ts   = f['photon_data/timestamps'][:]
        nt   = f['photon_data/nanotimes'][:]
        mask = np.zeros(len(det), dtype=bool)
        for d in keep:
            mask |= (det == d)
        print(f'Keeping {mask.sum()} / {len(mask)} photons '
              f'({mask.sum()/len(mask)*100:.1f}%) — detectors {keep}')
        for key, arr in [('detectors', det), ('timestamps', ts), ('nanotimes', nt)]:
            del f[f'photon_data/{key}']
            f.create_dataset(f'photon_data/{key}', data=arr[mask])
    print(f'Filtered HDF5 → {output_h5}')


## 4 · Burst search + selection

**Fixed parameters (aligned with `calculate_BVA.py`):**
- Background: `exp_hist_fit`, 60s, F_bg=1.7 *(was: exp_fit, 30s)*
- Burst search: `m=15`, `Ph_sel(Dex='DAem')` *(was: m=20, all photons)*
- Selection: naa≥30 → size≥150 → S∈[0.25,0.85] *(was: size≥60, naa≥40, no S filter)*


In [ ]:
def load_and_burst_search(
    h5_file: str,
    D_ON: tuple = (0, 1000),
    A_ON: tuple = (1000, 2000),
    m: int = 15,           # FIX: was 20 — reference uses 15
    F: float = 6.0,
    min_naa: int = 30,     # FIX: was 40 — reference uses 30
    min_burst_size: int = 150,   # FIX: was 60 — reference uses 150
    S1: float = 0.25,      # FIX: was missing — reference uses 0.25
    S2: float = 0.85,      # FIX: was missing — reference uses 0.85
):
    """
    Load Photon-HDF5, apply ALEX periods, run burst search + 3-stage selection.

    Parameters aligned with calculate_BVA.py:
      bg.exp_hist_fit, 60s, F_bg=1.7
      m=15 on Ph_sel(Dex='DAem')
      naa >= 30  →  size >= 150  →  S in [0.25, 0.85]
    """
    data = frb.loader.photon_hdf5(h5_file)
    data.add(
        donor=1, acceptor=0,
        alex_period_donor=D_ON,
        alex_period_acceptor=A_ON,
    )
    frb.loader.alex_apply_period(data)

    # FIX: exp_hist_fit + 60s + F_bg=1.7  (was: exp_fit, 30s)
    data.calc_bg(fun=frb.bg.exp_hist_fit, time_s=60, tail_min_us='auto', F_bg=1.7)

    # FIX: Ph_sel(Dex='DAem') + m=15  (was: all photons, m=20)
    data.burst_search(m=m, F=F, ph_sel=frb.Ph_sel(Dex='DAem'), computefret=False)
    data.calc_fret(count_ph=True, corrections=False)

    # 3-stage selection from calculate_BVA.py
    data_sel = data.select_bursts(frb.select_bursts.naa,  th1=min_naa,        computefret=False)
    data_sel = data_sel.select_bursts(frb.select_bursts.size, th1=min_burst_size, computefret=False)
    data_sel = data_sel.select_bursts(frb.select_bursts.S, S1=S1, S2=S2,      computefret=False)  # FIX: was missing

    print(f'Total bursts             : {data.mburst[0].num_bursts}')
    print(f'After naa >= {min_naa}         : {data.select_bursts(frb.select_bursts.naa, th1=min_naa, computefret=False).mburst[0].num_bursts}')
    print(f'After size >= {min_burst_size}     : {data.select_bursts(frb.select_bursts.naa, th1=min_naa, computefret=False).select_bursts(frb.select_bursts.size, th1=min_burst_size, computefret=False).mburst[0].num_bursts}')
    print(f'After S in [{S1},{S2}]  : {data_sel.mburst[0].num_bursts}')

    return data, data_sel


## 5 · BVA helpers

**Fixed:** chunk_size = 7 *(was 5)*, NaN-safe `bin_bva`.


In [ ]:
# FIX: chunk_size=7 (was 5) — aligned with calculate_BVA.py (n=7)
BVA_CHUNK_SIZE = 7

def BVA(data, chunk_size: int = BVA_CHUNK_SIZE):
    """BVA: sub-burst FRET std-dev per burst. chunk_size=7 from calculate_BVA.py"""
    E_eff, std_E = [], []
    for ich, mburst in enumerate(data.mburst):
        stdE, E = [], []
        Aem = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='Aem'))
        Dex = data.get_ph_mask(ich=ich, ph_sel=frb.Ph_sel(Dex='DAem'))
        for istart, istop in zip(mburst.istart, mburst.istop):
            phots = Aem[istart:istop+1][Dex[istart:istop+1]]
            Esub  = [phots[nb:ne].sum()
                     for nb, ne in zip(range(0, phots.size, chunk_size),
                                       range(chunk_size, phots.size+1, chunk_size))]
            if Esub:
                stdE.append(np.std(Esub) / chunk_size)
                E.append(sum(Esub) / (len(Esub) * chunk_size))
            else:
                stdE.append(np.nan)
                E.append(np.nan)
        E_eff.append(np.array(E))
        std_E.append(np.array(stdE))
    return E_eff, std_E


def bin_bva(E, std, R: int = 10, B_thr: int = 50):
    """Bin BVA results — NaN-safe version."""
    E_cat   = np.concatenate(E)
    std_cat = np.concatenate(std)
    valid   = np.isfinite(E_cat) & np.isfinite(std_cat)  # FIX: NaN guard
    E_cat, std_cat = E_cat[valid], std_cat[valid]
    bn = np.linspace(0, 1, R + 1)
    std_avg, E_avg = np.full(R, -1.0), np.full(R, -1.0)
    for i, (bb, be) in enumerate(zip(bn[:-1], bn[1:])):
        mask = (bb <= E_cat) & (E_cat < be)
        if mask.sum() > B_thr:
            std_avg[i] = np.mean(std_cat[mask])
            E_avg[i]   = np.mean(E_cat[mask])
    return E_avg, std_avg


# Shot-noise reference — chunk_size=7
x_T = np.arange(0, 1.01, 0.01)
y_T = np.sqrt((x_T * (1 - x_T)) / BVA_CHUNK_SIZE)


## 6 · User input — PTU file paths

In [ ]:
from pathlib import Path

ptu_file    = r'C:/Users/fatma/Dropbox/2026/PhD experiments/SmFRET/2605/260517_sFRET_TFAE_3a_4a_mutants.sptw/260517_sFRET_TFAE_3a_4000x_apo1.ptu'
_ptu        = Path(ptu_file)
output_h5   = str(_ptu.with_suffix('.hdf5'))
filtered_h5 = str(_ptu.with_stem(_ptu.stem + '_filtered').with_suffix('.hdf5'))

sample_name = ''     # e.g. 'APO', 'HOLO'
buffer_name = 'Tris'
author      = 'Fatma'

ptu_to_hdf5(
    ptu_file=ptu_file,
    output_path=output_h5,
    description='smFRET — Alexa555/Alexa647',
    author=author,
    author_affiliation='KUL',
    sample_name=sample_name,
    buffer_name=buffer_name,
    dye_names='Alexa555, Alexa647',
)


## 7 · Filter junk detectors

In [ ]:
filter_detectors(output_h5, filtered_h5, keep=(0, 1))

## 8 · Run burst search

In [ ]:
smfret_data, smfret_data_sel = load_and_burst_search(
    h5_file=filtered_h5,
    D_ON=(0, 1000),
    A_ON=(1000, 2000),
    m=15,          # aligned with calculate_BVA.py
    F=6.0,
    min_naa=30,    # aligned with calculate_BVA.py
    min_burst_size=150,   # aligned with calculate_BVA.py
    S1=0.25,       # aligned with calculate_BVA.py
    S2=0.85,       # aligned with calculate_BVA.py
)

print(f'Acquisition duration   : {smfret_data.acquisition_duration:.1f} s')
print(f'Total bursts           : {smfret_data.mburst[0].num_bursts}')
print(f'Bursts after selection : {smfret_data_sel.mburst[0].num_bursts}')

frb.alex_jointplot(smfret_data_sel)
plt.title('After burst search + naa + size + S filter')
plt.show()


## 9 · First-round H²MM — identify donor-only bursts

Fit 1–6 state models. Check ICL and BICp plots to pick the optimal number of states.


In [ ]:
print('Fitting first-round H2MM (may take a few minutes)...')
smfret_data_mp = bhm.BurstData(smfret_data_sel)
smfret_data_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

# ICL plot
smfret_data_mp.models.find_ideal('ICL', auto_set=True)
bhm.ICL_plot(smfret_data_mp.models, highlight_ideal=True)
plt.title('ICL — first round (look for minimum/elbow)')
plt.tight_layout(); plt.show()

# BICp plot
smfret_data_mp.models.find_ideal('BICp', auto_set=False)
bhm.BICp_plot(smfret_data_mp.models, highlight_ideal=True)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — first round (pick first model below red line)')
plt.tight_layout(); plt.show()


In [ ]:
# ── SET based on ICL/BICp plots above (0-indexed) ─────────────────────────
model_id_round1 = 3   # ← adjust: 0=1-state, 1=2-state, 2=3-state ...
# ─────────────────────────────────────────────────────────────────────────

model_r1 = smfret_data_mp.models[model_id_round1]
print(f'States : {model_r1.nstate}')
print(f'E      : {model_r1.E}')
print(f'S      : {model_r1.S}')

bhm.dwell_ES_scatter(model_r1, alpha=0.2, s=2)
bhm.trans_arrow_ES(model_r1)
plt.title(f'First-round H2MM — {model_r1.nstate} states')
plt.tight_layout(); plt.show()


## 10 · Remove donor-only bursts

**FIX:** The original code used hardcoded `burst_type` bitmasks (0b0010, 0b1000, 0b1010)
which are unreliable — the correct approach is to identify the donor-only state
by its high S value and remove bursts that only visit that state.


In [ ]:
# Identify donor-only state = state with highest S value (low E, high S)
donor_only_state = int(np.argmax(model_r1.S))
print(f'Donor-only state: {donor_only_state}  '
      f'E={model_r1.E[donor_only_state]:.3f}  '
      f'S={model_r1.S[donor_only_state]:.3f}')

# Keep bursts that have at least one photon NOT assigned to donor-only state
# FIX: was using unreliable burst_type bitmask
fret_mask = np.array([
    not np.all(path == donor_only_state)
    for path in model_r1.path
])
lf_hf_idxs = np.where(fret_mask)

print(f'Bursts before H2MM filter : {len(fret_mask)}')
print(f'Bursts after  H2MM filter : {fret_mask.sum()}')

# E histogram comparison
fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(smfret_data_sel.E[0], bins=50, density=True,
        histtype='step', label='Before filter')
ax.hist(smfret_data_sel.E[0][lf_hf_idxs[0]], bins=50, density=True,
        histtype='step', label='After donor-only removal')
ax.set_xlim(0, 1)
ax.set_xlabel(r'$E$')
ax.set_ylabel('Norm. density')
ax.legend()
plt.tight_layout(); plt.show()

smfret_data_hmm = smfret_data_sel.select_bursts_mask_apply(lf_hf_idxs)
frb.alex_jointplot(smfret_data_hmm)
plt.title('FRET-active bursts after donor-only removal')
plt.show()


## 11 · Second-round H²MM — final model on FRET-active bursts

In [ ]:
print('Fitting second-round H2MM on FRET-active bursts...')
smfret_data_hmm_mp = bhm.BurstData(smfret_data_hmm)
smfret_data_hmm_mp.models.calc_models(max_state=6, max_iter=3600)
print('Done.')

smfret_data_hmm_mp.models.find_ideal('ICL', auto_set=True)
bhm.ICL_plot(smfret_data_hmm_mp.models, highlight_ideal=True)
plt.title('ICL — second round')
plt.tight_layout(); plt.show()

smfret_data_hmm_mp.models.find_ideal('BICp', auto_set=False)
bhm.BICp_plot(smfret_data_hmm_mp.models, highlight_ideal=True)
plt.axhline(0.005, color='red', linestyle='--', label='threshold 0.005')
plt.legend()
plt.title('BICp — second round')
plt.tight_layout(); plt.show()


In [ ]:
# ── SET based on second-round plots ───────────────────────────────────────
model_id_final = 2   # ← adjust
# ─────────────────────────────────────────────────────────────────────────

model_final = smfret_data_hmm_mp.models[model_id_final]
print(f'Final states : {model_final.nstate}')
print(f'E            : {model_final.E}')
print(f'S            : {model_final.S}')
print(f'Trans matrix :\n{model_final.trans}')

fig, ax = plt.subplots(figsize=(5, 4))
bhm.dwell_ES_scatter(model_final, alpha=0.6, s=1)
bhm.trans_arrow_ES(model_final, min_rate=0)
bhm.scatter_ES(model_final, s=100, c='r')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_xlabel(r'$E$'); ax.set_ylabel(r'$S$')
plt.title(f'Final H2MM — {model_final.nstate} states')
plt.tight_layout(); plt.show()


## 12 · Dwell-time extraction and survival plots

In [ ]:
model = smfret_data_hmm_mp.models[model_id_final]
n_states = model.nstate

# Try modern burstH2MM API first, fall back to dwell_pos
try:
    dwell_pos = model.dwell_pos
    print('dwell_pos shape:', dwell_pos.shape)
except AttributeError:
    print('dwell_pos not available — attributes:',
          [k for k in model.__dict__ if 'dwell' in k.lower()])

# Mean photon rate for photon→time conversion
bursts = smfret_data_hmm.mburst[0]
mean_ph_rate = bursts.counts.sum() / (bursts.width.sum() * smfret_data_hmm.clk_p)
print(f'Mean in-burst photon rate: {mean_ph_rate:.0f} ph/s')

fig, axes = plt.subplots(1, n_states, figsize=(4*n_states, 4), sharey=True)
if n_states == 1:
    axes = [axes]

for state_id, ax in enumerate(axes):
    try:
        dwell_t = model.dwell_times[state_id]
        xlabel  = 'Dwell time (s)'
    except (AttributeError, IndexError):
        raw = dwell_pos[dwell_pos[:, 3] == state_id]
        dwell_t = (raw[:, 2] - raw[:, 1]) / mean_ph_rate
        xlabel  = 'Dwell time (s, estimated)'

    dwell_t = dwell_t[dwell_t > 0]
    t_sort  = np.sort(dwell_t)
    surv    = 1 - np.arange(1, len(t_sort)+1) / len(t_sort)

    ax.semilogy(t_sort, surv, '.', markersize=3)
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Survival probability')
    ax.set_title(f'State {state_id}  E≈{model.E[state_id]:.2f}  (n={len(dwell_t)})')

plt.suptitle('Dwell-time survival per H²MM state')
plt.tight_layout(); plt.show()


## 13 · BVA — Burst Variance Analysis

chunk_size = 7 (aligned with `calculate_BVA.py`).  
Points above the dashed shot-noise line indicate genuine conformational dynamics.


In [ ]:
E_bva, std_bva = BVA(smfret_data_hmm, chunk_size=BVA_CHUNK_SIZE)
E_avg, std_avg = bin_bva(E_bva, std_bva, R=10, B_thr=50)

E_cat   = np.concatenate(E_bva)
std_cat = np.concatenate(std_bva)
valid   = np.isfinite(E_cat) & np.isfinite(std_cat)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(E_cat[valid], std_cat[valid], s=2, alpha=0.3, color='grey', label='per burst')

valid_bins = E_avg > 0
ax.scatter(E_avg[valid_bins], std_avg[valid_bins], s=40, color='blue',
           zorder=5, label='binned avg')

ax.plot(x_T, y_T, 'k--', lw=2, label=f'shot noise (n={BVA_CHUNK_SIZE})')
ax.set_xlim(0, 1)
ax.set_ylim(0, np.sqrt(0.5**2 / BVA_CHUNK_SIZE) * 2)
ax.set_xlabel(r'$\langle E \rangle$')
ax.set_ylabel(r'$\sigma_E$')
ax.text(0.05, 0.95, f'n = {valid.sum()} bursts', va='top', transform=ax.transAxes)
ax.legend()
plt.title(f'BVA (n={BVA_CHUNK_SIZE}) — above dashed line = dynamics')
plt.tight_layout(); plt.show()
